# Extrator do Lacuna com QLoRA

Este notebook ajusta um modelo pequeno (Qwen2.5-Instruct) para ler um extrato de contrato de diário oficial e devolver o JSON do esquema `Ato` do Lacuna: tipo, número, processo, contratada, CNPJ, objeto, valor e data de assinatura.

A pergunta é se um modelo de 1,5B parâmetros, ajustado com 289 exemplos, chega perto do Claude Opus 5.5 no mesmo conjunto ouro, rodando de graça numa GPU T4.

- **Treino:** `data/treino/rotulos_claude.jsonl`. São 289 trechos de 13 municípios do RJ rotulados por LLM e sem revisão humana. Ficaram de fora os trechos com o mesmo texto ou o mesmo contrato do conjunto ouro.
- **Avaliação:** `data/gold/dev.jsonl` (40) e `data/gold/test.jsonl` (60), rótulos revisados à mão, com a mesma função `score` usada para o regex e para os LLMs via API.
- **Ambiente:** Google Colab com GPU T4 (Ambiente de execução → Alterar o tipo → T4 GPU).

In [ ]:
!git clone -q https://github.com/juan-barross/lacuna.git
%cd lacuna
!pip install -q -e . "transformers>=4.56" "peft>=0.13" "trl>=0.20" "bitsandbytes>=0.44" accelerate datasets

## Configuração

Todo o experimento depende destes valores. Cada execução grava uma linha em `runs/historico.jsonl` com a configuração e o resultado.

In [ ]:
BASE = "Qwen/Qwen2.5-1.5B-Instruct"
RUN = "qwen2.5-1.5b-r16-e3"
SEED = 2027
AVALIAR_MODELO_SEM_AJUSTE = True
USUARIO_HF = "SEU_USUARIO_HF"

LORA = dict(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
TREINO = dict(
    num_train_epochs=3,
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
)

## Dados

O alvo de cada exemplo é o JSON que o `LLMExtractor` pede ao modelo via API, e o prompt de sistema é o mesmo `PROMPT`. Assim a comparação com o Opus e o Nemotron mede só o modelo.

In [ ]:
import json
import sys

sys.path.insert(0, "src")

from datasets import Dataset

from dxp.evaluation import score
from dxp.llm import PROMPT
from dxp.schema import Ato


def ler(path):
    return [json.loads(line) for line in open(path, encoding="utf-8")]


def pergunta(texto):
    return [{"role": "system", "content": PROMPT}, {"role": "user", "content": texto}]


def alvo(row):
    return json.dumps(Ato.model_validate(row).model_dump(mode="json"), ensure_ascii=False)


treino = ler("data/treino/rotulos_claude.jsonl")
ouro = {split: ler(f"data/gold/{split}.jsonl") for split in ("dev", "test")}
assert not {r["text"] for r in treino} & {r["text"] for g in ouro.values() for r in g}, (
    "trecho do ouro no treino"
)

exemplos = [
    {"prompt": pergunta(r["text"]), "completion": [{"role": "assistant", "content": alvo(r)}]} for r in treino
]
ds = Dataset.from_list(exemplos).train_test_split(test_size=0.1, seed=SEED)
print(ds)
print(exemplos[0]["completion"][0]["content"])

## Modelo em 4 bits

O QLoRA carrega o modelo base quantizado em NF4 e treina só os adaptadores LoRA. Na T4 não há bf16, por isso o cálculo é em fp16.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

torch.manual_seed(SEED)
tok = AutoTokenizer.from_pretrained(BASE)
quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE, quantization_config=quant, device_map="auto", dtype=torch.float16
)

tamanhos = sorted(
    len(tok.apply_chat_template(e["prompt"] + e["completion"], return_dict=True)["input_ids"])
    for e in exemplos
)
print(f"tokens por exemplo: mediana {tamanhos[len(tamanhos) // 2]}, máximo {tamanhos[-1]}")
print(f"memória da GPU: {torch.cuda.memory_allocated() / 2**30:.1f} GiB")

## Como o extrator é medido

A saída é lida como JSON do `Ato`. Quando não é JSON válido, o trecho vira `Ato(tipo="outro")`, a mesma regra do extrator via API, e conta como erro. A geração é gulosa (sem amostragem), então o resultado se repete.

In [ ]:
import statistics
import time

from pydantic import ValidationError


def so_json(resposta):
    inicio, fim = resposta.find("{"), resposta.rfind("}")
    return resposta[inicio : fim + 1] if inicio != -1 else resposta


class Extrator:
    def __init__(self, model, max_new_tokens=512):
        self.model, self.max_new_tokens = model, max_new_tokens
        self.erros, self.segundos = 0, []

    def __call__(self, texto):
        entrada = tok.apply_chat_template(
            pergunta(texto), add_generation_prompt=True, return_tensors="pt", return_dict=True
        ).to(self.model.device)
        inicio = time.perf_counter()
        with torch.no_grad():
            saida = self.model.generate(
                **entrada, max_new_tokens=self.max_new_tokens, do_sample=False, pad_token_id=tok.eos_token_id
            )
        self.segundos.append(time.perf_counter() - inicio)
        resposta = tok.decode(saida[0, entrada["input_ids"].shape[1] :], skip_special_tokens=True)
        try:
            return Ato.model_validate_json(so_json(resposta))
        except ValidationError:
            self.erros += 1
            return Ato(tipo="outro")

    def uso(self, nome):
        s = sorted(self.segundos)
        return {
            "modelo": nome,
            "gpu": torch.cuda.get_device_name(),
            "chamadas": len(s),
            "erros": self.erros,
            "custo_total_usd": 0.0,
            "latencia_p50_s": round(statistics.median(s), 2),
            "latencia_p95_s": round(s[int(0.95 * (len(s) - 1))], 2),
        }


def avaliar(model, split, nome):
    model.eval()
    extrator = Extrator(model)
    campos = score(ouro[split], extrator)
    return {"campos": campos, "uso": extrator.uso(nome)}


def resumo(resultado):
    return {campo: round(m["f1"], 2) for campo, m in resultado["campos"].items()}

In [ ]:
resultados = {}
if AVALIAR_MODELO_SEM_AJUSTE:
    resultados["base_test"] = avaliar(model, "test", BASE)
    print(resumo(resultados["base_test"]), resultados["base_test"]["uso"])

## Treino

A perda é calculada só na resposta (`completion`), não no prompt. 10% do treino fica separado para acompanhar a perda de validação a cada época.

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
    output_dir=f"runs/{RUN}",
    seed=SEED,
    max_length=2048,
    fp16=True,
    gradient_checkpointing=True,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    **TREINO,
)
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    processing_class=tok,
    peft_config=LoraConfig(task_type="CAUSAL_LM", **LORA),
)
trainer.model.print_trainable_parameters()
inicio = time.perf_counter()
trainer.train()
minutos_treino = round((time.perf_counter() - inicio) / 60, 1)
print(f"treino: {minutos_treino} min")

In [ ]:
perdas = [(r["epoch"], r["eval_loss"]) for r in trainer.state.log_history if "eval_loss" in r]
print("perda de validação por época:", perdas)

## Avaliação no conjunto ouro

O dev serve para comparar configurações. O test só entra no relatório final.

In [ ]:
from datetime import date
from pathlib import Path

nome = f"{BASE} + LoRA ({RUN})"
for split in ("dev", "test"):
    resultados[split] = avaliar(trainer.model, split, nome)
    print(split, resumo(resultados[split]), resultados[split]["uso"])
    Path("reports").mkdir(exist_ok=True)
    Path(f"reports/{date.today()}_qlora_{split}.json").write_text(
        json.dumps(resultados[split], indent=2, ensure_ascii=False), encoding="utf-8"
    )

## Comparação com os outros extratores

Lê os relatórios que já estão em `reports/` (regex, Opus, Nemotron) e põe o F1 de cada campo lado a lado.

In [ ]:
import pandas as pd

linhas = {}
for path in sorted(Path("reports").glob("*_test.json")):
    rel = json.loads(path.read_text(encoding="utf-8"))
    if "campos" in rel:
        linhas[path.stem.split("_", 1)[1]] = resumo(rel)
if "base_test" in resultados:
    linhas["base sem ajuste_test"] = resumo(resultados["base_test"])
pd.DataFrame(linhas).T

## Histórico de execuções

Uma linha por execução, com a configuração e o F1 do dev e do test. É o que permite comparar r=8 com r=16, 0,5B com 1,5B e assim por diante.

In [ ]:
import peft
import transformers
import trl

linha = {
    "run": RUN,
    "data": str(date.today()),
    "base": BASE,
    "seed": SEED,
    "lora": LORA,
    "treino": TREINO,
    "exemplos_treino": len(ds["train"]),
    "minutos_treino": minutos_treino,
    "perda_validacao": perdas,
    "f1_dev": resumo(resultados["dev"]),
    "f1_test": resumo(resultados["test"]),
    "erros_json": {s: resultados[s]["uso"]["erros"] for s in ("dev", "test")},
    "versoes": {"transformers": transformers.__version__, "peft": peft.__version__, "trl": trl.__version__},
}
Path("runs").mkdir(exist_ok=True)
with open("runs/historico.jsonl", "a", encoding="utf-8") as f:
    f.write(json.dumps(linha, ensure_ascii=False) + "\n")
linha

## Salvar e publicar

O adaptador tem poucos MB. Primeiro ele é salvo localmente e os relatórios vão para download, para entrarem no repositório. Depois vai para o Hugging Face como **privado**: confira o model card antes de tornar público.

O login usa o seu token do Hugging Face (Settings → Access Tokens, permissão *write*).

In [ ]:
from google.colab import files

trainer.model.save_pretrained(f"adapters/{RUN}")
tok.save_pretrained(f"adapters/{RUN}")

!zip -qr resultados_{RUN}.zip reports runs/historico.jsonl

files.download(f"resultados_{RUN}.zip")

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
REPO = f"{USUARIO_HF}/lacuna-extrator-qwen2.5-1.5b-lora"

trainer.model.push_to_hub(REPO, private=True)
tok.push_to_hub(REPO, private=True)